<center>

<h1 >Projet : Analyse de données e-commerce avec Python</h1>

<p><em>Mael Garnier - p2510998 - Polytech Lyon - 3A cycle informatique</em></p>
<p><em>octobre 2025</em></p>

</center>

In [1]:

import sys

# Importation des bibliothèques nécessaires
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

## Partie 1 : Génération d'un Dataset e-commerce

Nous allons générer l'équivalent d'une vente par jours de 2021 à 2024, pour chaque pays. Les données relatives aux produits (nom du produit, catégorie, prix, quantité) sont générées aléatoirement.

Ces données sont ensuites converties en DataFrame, puis écrites dans un fichier CSV.

In [3]:
dates = []
# Nombre de jours entre le 1er janvier 2021 et le 31 décembre 2024
totalDays = (datetime(2024, 12, 31) - datetime(2021, 1, 1)).days
# Pour chaque jour, on ajoute la date correspondante à la liste
for i in range (totalDays):
    dates.append(datetime(2021, 1, 1) + timedelta(days=i))

# Liste de pays
countries = ['France', 'Germany', 'USA', 'India', 'Japan']

# Liste de produits, par catégorie
categories = ['Electronics', 'Clothes', 'Sport']
products_by_cat = {
    'Electronics': ['Laptop', 'Smartphone', 'Headphones', 'Camera'],
    'Clothes': ['T-shirt', 'Dress', 'Shoes'],
    'Sport': ['Surf', 'Bike', 'Skate'],
}

# On génère un prix pour chaque produit
price_for_product = {}
for (cat, products) in products_by_cat.items():
    for product in products:
        price_for_product[product] = random.randint(50, 2000)

# On génère les données, en attribuant une vente par jour par pays
data = []
for country in countries:
    for day in dates:
        category = random.choice(categories) # Catégorie aléatoire
        product = random.choice(products_by_cat[category]) # Produit aléatoire dans la liste
        quantity = random.randint(1, 20) # Quantité aléatoire entre 1 et 20
        price = price_for_product[product] # On récupère le prix du produit, précédemment généré aléatoirement
        data.append({
            'Country': country,
            'Date': day,
            'Category': category,
            'Product': product,
            'Quantity': quantity,
            'Price': price,
            'Total': price * quantity,
        })

# Création du DataFrame pandas
temp_df = pd.DataFrame(data)

# Affichage des données
print('5 premières lignes :')
print(temp_df.head())
print('\n Nombre total de lignes : ', temp_df.shape)

# On exporte dans un fichier CSV
temp_df.to_csv('dataset.csv', index=False)


5 premières lignes :
  Country       Date Category  Product  Quantity  Price  Total
0  France 2021-01-01  Clothes    Dress        17   1946  33082
1  France 2021-01-02  Clothes    Shoes        15   1204  18060
2  France 2021-01-03  Clothes  T-shirt        14   1406  19684
3  France 2021-01-04    Sport     Surf         9   1717  15453
4  France 2021-01-05    Sport    Skate        18    425   7650

 Nombre total de lignes :  (7300, 7)


## Partie 2 : Analyse de données e-commerce

Dans cette partie, nous allons nous concentrer sur l'analyse et le traitement des données precedemment générées.

### Gestion de fichier
Commencons par charger notre fichier CSV (en vérifiant les erreurs) et filtrer les ventes.

In [7]:
# Charger un fichier CSV dans un DataFrame et gérer les erreurs potentielles
def load_csv(file_name):
    try :
        # Lecture du fichier avec panda
        df = pd.read_csv(file_name)
        df['Date'] = pd.to_datetime(df['Date'])
        return df
    except FileNotFoundError:
        # Affichage du message d'erreur si le fichier n'est pas trouvé
        print("Le fichier du dataset n'est pas trouvé, merci de verifier que celui-ci se trouve à la racine du projet.")
        # On arrête l'execution, pour éviter de futures erreurs
        sys.exit(1)
    except pd.errors.ParserError:
        # Affichage du message d'erreur si le fichier est mal formaté
        print("Le fichier semble mal formaté.")
        # On arrête l'execution, pour éviter de futures erreurs
        sys.exit(1)
    except TypeError:
        print("La date n'est pas au bon format.")
        sys.exit(1)

# Filtrer un DataFrame pour conserver tous les elements entre deux dates données
def filter_df_by_date(df, start_date, end_date):
    # Filtrage des ventes entre deux dates données
    date_mask = (df['Date'] > start_date) & (df['Date'] <= end_date)
    return df.loc[date_mask]

# Effectue le même filtrage que la fonction précédente, mais de manière manuelle (sans utiliser de masques)
def filter_df_by_date_manual(df, start_date, end_date):
    filtered_df = pd.DataFrame()
    for index, row in df.iterrows():
        if start_date < row['Date'] <= end_date:
            filtered_df = pd.concat([filtered_df, row.to_frame().T], ignore_index=True)
    return filtered_df

dataframe = load_csv('dataset.csv')
print("Dataframe chargé : ", dataframe.head())
dataframe = filter_df_by_date(dataframe, datetime(2021, 1, 2), datetime(2024, 12, 31))
print("Dataframe filtré par date : ", dataframe.head())
dataframe = filter_df_by_date_manual(dataframe, datetime(2021, 1, 3), datetime(2024, 12, 31))
print("Dataframe filtré par date manuellement : ", dataframe.head())

Dataframe chargé :    Country       Date Category  Product  Quantity  Price  Total
0  France 2021-01-01  Clothes    Dress        17   1946  33082
1  France 2021-01-02  Clothes    Shoes        15   1204  18060
2  France 2021-01-03  Clothes  T-shirt        14   1406  19684
3  France 2021-01-04    Sport     Surf         9   1717  15453
4  France 2021-01-05    Sport    Skate        18    425   7650
Dataframe filtré par date :    Country       Date Category  Product  Quantity  Price  Total
2  France 2021-01-03  Clothes  T-shirt        14   1406  19684
3  France 2021-01-04    Sport     Surf         9   1717  15453
4  France 2021-01-05    Sport    Skate        18    425   7650
5  France 2021-01-06  Clothes  T-shirt         7   1406   9842
6  France 2021-01-07  Clothes  T-shirt         7   1406   9842
Dataframe filtré par date manuellement :    Country                 Date Category  Product Quantity Price  Total
0  France  2021-01-04 00:00:00    Sport     Surf        9  1717  15453
1  France  

### Analyse de données

Nous allons calculer : le chiffre d'affaires total, moyen, maximal, minimal sur la periode donnée.

Nous allons egalement identifier le top 3 des produits les plus vendus et le top 3 des pays les plus rentables sur la periode donnée.

Pour finir, nous allons detecter les potentielles anomalies de ventes (jours sans vente, pics de vente inhabituels).

In [22]:
dataframe = load_csv('dataset.csv')
dataframe = filter_df_by_date(dataframe, datetime(2022, 1, 1), datetime(2023, 12, 31))
ca_total = dataframe['Total'].sum()
ca_moyen = dataframe['Total'].mean()
ca_max = dataframe['Total'].max()
ca_min = dataframe['Total'].min()

print("Chiffre d'affaire total de 2022 à 2023 : ", ca_total)
print("Chiffre d'affaire moyen de 2022 à 2023 : ", ca_moyen)
print("Chiffre d'affaire maximal de 2022 à 2023 : ", ca_max)
print("Chiffre d'affaire minimal de 2022 à 2023 : ", ca_min)

def get_top_n_product_by_sales(df, n=3):
    # On calcul le total de ventes par produit
    top_products = []

    for index, row in df.iterrows():
        # Si le produit n'est pas encore dans la liste, on l'ajoute
        if not any(prod['Name'] == row['Product'] for prod in top_products):
            top_products.append({
                'Name': row['Product'],
                'Category': row['Category'],
                'Sales': row['Quantity'],
            })
        else:
            for product in top_products:
                if product['Name'] == row['Product']:
                    product['Sales'] += row['Quantity']
                    break

    # On rends top_products avec le nombre de produit donné par n
    top_products = sorted(top_products, key=lambda x: x['Sales'], reverse=True)[:n]
    return top_products

top_3_products = get_top_n_product_by_sales(dataframe, 3)

print("Top 3 des produits les plus vendus : ", top_3_products)

def get_top_n_countries(df, n=3):
    # On calcul le total de ventes par produit
    top_countries = []

    for index, row in df.iterrows():
        # Si le produit n'est pas encore dans la liste, on l'ajoute
        if not any(country['Country'] == row['Country'] for country in top_countries):
            top_countries.append({
                'Country': row['Country'],
                'CA': row['Total'],
            })
        else:
            for country in top_countries:
                if country['Country'] == row['Country']:
                    country['CA'] += row['Total']
                    break

    # On rends top_countries avec le nombre de pays donné par n
    top_countries = sorted(top_countries, key=lambda x: x['CA'], reverse=True)[:n]
    return top_countries

top_3_countries = get_top_n_countries(dataframe, 3)

print("Top 3 des pays les plus rentables : ", top_3_countries)


Chiffre d'affaire total de 2022 à 2023 :  50619885
Chiffre d'affaire moyen de 2022 à 2023 :  13887.485596707818
Chiffre d'affaire maximal de 2022 à 2023 :  38920
Chiffre d'affaire minimal de 2022 à 2023 :  425
Top 3 des produits les plus vendus :  [{'Name': 'Skate', 'Category': 'Sport', 'Sales': 4552}, {'Name': 'Bike', 'Category': 'Sport', 'Sales': 4532}, {'Name': 'Dress', 'Category': 'Clothes', 'Sales': 4438}]
Top 3 des pays les plus rentables :  [{'Country': 'Japan', 'CA': 10344906}, {'Country': 'France', 'CA': 10294878}, {'Country': 'India', 'CA': 10160863}]
